In [ ]:
# -*- coding: utf-8 -*-
"""
==============================================================================
 Fuzzy Cognitive Map (FCM) and Scenario Analysis for a
 Multilevel Failure & Instability Model (v2)
 in Educational Robotics (STEM) Interventions
 -- UPDATED to match the new 7-level thematic table (209 coded occurrences,
    21 sub-themes) and the new theoretical framework document
    ("منطق کلی چارچوب").

 INPUT FILE (docx) must contain two tables (order-independent -- detected
 automatically by column count):
   - "Full" table  (5 columns): main theme | sub-theme | concepts+freq | sources | total freq
   - "Summary" table (3 columns): main theme | sub-theme | total freq

 KEY STRUCTURAL CHANGE vs. the old model:
   - There are now SIX causal/content levels (L1..L6, ecological, micro->macro)
     whose frequencies sum to 176, PLUS a seventh "meta-evidence" dimension
     (L7, freq=33) that the theory document explicitly says must NOT be
     modeled as a causal contributor (it reflects limits on *observing/
     measuring* failure, not a cause of it). We honour that: L7 only
     RECEIVES weight from its own sub-themes (Receiver role) and has NO
     outgoing causal edge into the rest of the network. Its relationship
     to the outcome is drawn as a dashed, non-quantitative annotation only.
   - Two feedback loops named explicitly in the theory doc are now encoded:
       * "Capacity-erosion loop": learner-level erosion (L2) feeds back and
         further undermines teacher capacity (L1).
       * "Support-erosion loop": the breakdown/outcome node feeds back and
         erodes organizational resources (L5) and policy/leadership (L6).
==============================================================================
"""



In [ ]:
# %% [1] Install / import libraries -----------------------------------------
# !pip install python-docx scikit-learn networkx scipy pandas matplotlib -q

import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats
import docx

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.dpi"] = 300

DPI = 300  # Q1 print standard (300 dpi keeps print quality while staying
           # memory-safe for the larger 29-node network figures; raise back
           # to 600 locally if you have more RAM available)

def save_fig(fig, name):
    """Save both a 600-dpi PNG and a vector PDF, with no clipped labels."""
    fig.savefig(f"{name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.8)
    fig.savefig(f"{name}.pdf", bbox_inches="tight", pad_inches=0.8)
    plt.close(fig)



In [ ]:
# %% [2] Load input file --------------------------------------------------
# In Colab: use the file-upload widget, then set the path in THEMES_DOCX
#
#   from google.colab import files
#   uploaded = files.upload()
#
THEMES_DOCX = "جدول_نهایی_مضامین_شکست_و_ناپایداری_هم_راستا_با_مبانی_نظری.docx"

pdig = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
def to_num(s):
    s = str(s).translate(pdig).strip()
    return int(s) if s else None

d = docx.Document(THEMES_DOCX)
assert len(d.tables) >= 2, "Input file structure does not match the reference version."

# --- Detect tables by column count (robust to table order) ---
t_full, t_summary = None, None
for t in d.tables:
    ncols = len(t.columns)
    if ncols >= 5 and t_full is None:
        t_full = t
    elif ncols == 3 and t_summary is None:
        t_summary = t
assert t_full is not None and t_summary is not None, \
    "Could not auto-detect the full (5-col) and summary (3-col) tables."

# --- Summary table (official, precise numbers) ---
rows_summary, last_main = [], None
for ri, row in enumerate(t_summary.rows):
    if ri == 0:
        continue
    cells = [c.text.strip() for c in row.cells]
    main_theme = cells[0] if cells[0] else last_main
    if cells[0]:
        last_main = cells[0]
    rows_summary.append({"main_theme_raw": main_theme, "sub_theme": cells[1],
                          "total_freq": to_num(cells[2])})
df_summary = pd.DataFrame(rows_summary)

# --- Full table (concept text + sources, for TF-IDF analysis and documentation) ---
rows_full, last_main = [], None
for ri, row in enumerate(t_full.rows):
    if ri == 0:
        continue
    cells = [c.text.strip() for c in row.cells]
    main_theme = cells[0] if cells[0] else last_main
    if cells[0]:
        last_main = cells[0]
    rows_full.append({"main_theme_raw": main_theme, "sub_theme": cells[1],
                       "concepts_text": cells[2], "sources": cells[3]})
df_full = pd.DataFrame(rows_full)

records = df_full.merge(df_summary[["sub_theme", "total_freq"]], on="sub_theme", how="left")
records = records.to_dict("records")
print(f"Number of sub-themes extracted: {len(records)}")
print(f"Grand total frequency (should be 209): {df_summary['total_freq'].sum()}")



In [ ]:
# %% [3] Extract individual concepts (name, frequency, description) -------------
FREQ_PAT = re.compile(r"\(\s*(?:فراوانی:\s*)?([۰-۹0-9]+)\s*\):\s*")

def parse_concepts(text):
    parts = re.split(r"\s*/\s*(?=[^:]*?\(\s*(?:فراوانی:\s*)?[۰-۹0-9]+\s*\):)", text)
    out = []
    for p in parts:
        p = p.strip().strip(".").strip()
        matches = list(FREQ_PAT.finditer(p))
        if not matches:
            continue
        last = matches[-1]
        name = p[:last.start()].strip()
        freq = int(last.group(1).translate(pdig))
        rest = p[last.end():].strip()
        out.append({"name": name, "freq": freq, "desc": rest})
    return out

concepts = []
for r in records:
    for c in parse_concepts(r["concepts_text"]):
        c["main_theme_raw"] = r["main_theme_raw"]
        c["sub_theme"] = r["sub_theme"]
        concepts.append(c)
print(f"Number of individual concepts extracted (for text analysis): {len(concepts)}")



In [ ]:
# %% [4] Layer mapping: raw main theme <-> theoretical level number ------------
# Levels 1-6 are the causal/ecological content levels (micro -> macro);
# Level 7 is the META-EVIDENCE dimension (non-causal, see module docstring).
LAYER_KEYWORDS = [
    (1, "مجریان",          "Individual level (implementers/teachers): capacity & readiness deficits"),
    (2, "یادگیرندگان",     "Individual level (learners): cognitive, motivational & participatory vulnerabilities"),
    (3, "سطح آموزشی",      "Instructional level: curriculum & instructional-design misalignment"),
    (4, "فناورانه",        "Technological level: complexity, usability & fragility"),
    (5, "سازمانی",         "Organizational-implementation level: resource deficits & time pressure"),
    (6, "کلان",            "Macro/contextual level: policy, cultural & access constraints"),
    (7, "فراسطحی",         "Meta-evidence dimension: limits on detecting/evaluating failure (NON-CAUSAL)"),
]
def get_layer(main_theme_text):
    for lyr, kw, _ in LAYER_KEYWORDS:
        if kw in main_theme_text:
            return lyr
    raise ValueError(f"No layer found for: {main_theme_text}")

layer_name = {lyr: name for lyr, _, name in LAYER_KEYWORDS}
df_summary["layer"] = df_summary["main_theme_raw"].apply(get_layer)

CAUSAL_LAYERS = [1, 2, 3, 4, 5, 6]
META_LAYER = 7

check = df_summary.groupby("layer")["total_freq"].sum()
expected = {1: 17, 2: 61, 3: 31, 4: 14, 5: 40, 6: 13, 7: 33}
for lyr, exp in expected.items():
    got = check.get(lyr, None)
    flag = "OK" if got == exp else "!! MISMATCH !!"
    tag = " (meta, non-causal)" if lyr == META_LAYER else ""
    print(f"Layer {lyr}{tag} ({layer_name[lyr]}): computed frequency={got} | expected={exp} [{flag}]")

CAUSAL_TOTAL = int(df_summary[df_summary["layer"].isin(CAUSAL_LAYERS)]["total_freq"].sum())
GRAND_TOTAL = int(df_summary["total_freq"].sum())
print(f"\nCausal-level total (levels 1-6): {CAUSAL_TOTAL} (theoretical doc expects 176)")
print(f"Grand total incl. meta-evidence: {GRAND_TOTAL} (theoretical doc expects 209)")



In [ ]:
# %% [5] Per-sub-theme statistics (frequency and relative vulnerability weight) ---
df_sub = df_summary.copy()
df_sub["vuln_weight"] = df_sub["total_freq"] / df_sub["total_freq"].max()  # normalized to [0,1]
print(df_sub[["layer", "sub_theme", "total_freq", "vuln_weight"]].to_string(index=False))
df_sub.to_csv("table1_subtheme_stats.csv", index=False, encoding="utf-8-sig")

sub_row = {row.sub_theme: row for row in df_sub.itertuples()}
sub_themes = df_sub["sub_theme"].tolist()



In [ ]:
# %% [6] Define FCM nodes (6 causal layers + 1 meta layer + 21 sub-themes + 1 OUT) --
node_labels, label_map_layer, label_map_sub = {}, {}, {}
nid = 0
for lyr in range(1, 8):  # 1..6 causal + 7 meta
    node_labels[f"L{lyr}"] = layer_name[lyr]
    label_map_layer[lyr] = f"L{lyr}"
    nid += 1
for st in sub_themes:
    node_labels[f"S{nid}"] = st
    label_map_sub[st] = f"S{nid}"
    nid += 1
OUTCOME = "OUT"
node_labels[OUTCOME] = "Breakdown point / cascading failure & instability probability"
all_nodes = list(node_labels.keys())
n = len(all_nodes)
idx = {k: i for i, k in enumerate(all_nodes)}
print(f"\nTotal number of FCM nodes: {n}  "
      f"(6 causal layers + 1 meta layer + {len(sub_themes)} sub-themes + 1 output node)")



In [ ]:
# %% [7] Build the FCM weight matrix ---------------------------------------------------
W = np.zeros((n, n))

# (a) sub-theme -> its own layer edge: weight = relative frequency (always
#     positive). This applies to ALL 7 layers, including the meta layer --
#     it is a structural/compositional edge (how much each sub-theme
#     contributes to its parent dimension), not a causal claim.
for st in sub_themes:
    row = sub_row[st]
    W[idx[label_map_sub[st]], idx[label_map_layer[row.layer]]] = round(row.vuln_weight, 4)

# (b) directed cascading layer edges: outermost (macro, L6) -> innermost
#     (individual-implementer core, L1), among the SIX CAUSAL layers only.
#     Weight is based on each source layer's share of the causal total (176).
layer_total_freq = df_sub.groupby("layer")["total_freq"].sum()
cascade_pairs = [(6, 5), (5, 4), (4, 3), (3, 2), (2, 1)]
for src, dst in cascade_pairs:
    w = round(layer_total_freq[src] / CAUSAL_TOTAL, 4)
    W[idx[f"L{src}"], idx[f"L{dst}"]] = w

# (c) Capacity-erosion feedback loop (explicitly named in the theory document,
#     section 8): learner-level erosion (L2) feeds back and further
#     undermines teacher capacity & readiness (L1). Coefficient 0.5
#     moderates this indirect feedback relative to the main cascade.
w21 = round(0.5 * layer_total_freq[2] / CAUSAL_TOTAL, 4)
W[idx["L2"], idx["L1"]] += w21

# (d) core -> breakdown-point edge: only L1 (innermost, individual-implementer
#     capacity) connects directly to the outcome node; other causal layers
#     only affect it indirectly through the cascading chain (below).
w_core = round(layer_total_freq[1] / CAUSAL_TOTAL, 4)
W[idx["L1"], idx[OUTCOME]] = w_core

# (e) direct, weak edges from every OTHER causal layer to the breakdown
#     point (each layer's background/systemic contribution, independent of
#     the cascading chain; attenuated by distance from the L1 core).
core_distance = {2: 1, 3: 2, 4: 3, 5: 4, 6: 5}
for lyr, dist in core_distance.items():
    w = round((layer_total_freq[lyr] / CAUSAL_TOTAL) * (1 / (dist + 1)), 4)
    W[idx[f"L{lyr}"], idx[OUTCOME]] += w

# (f) Support-erosion feedback loop (explicitly named in the theory document,
#     section 9): poor outcomes / low participation feed back and erode
#     organizational resources (L5) and policy/leadership support (L6),
#     which in turn intensifies the demand-capacity mismatch. Small,
#     documented coefficient (0.2 each) relative to the forward cascade.
W[idx[OUTCOME], idx["L5"]] += round(0.2 * layer_total_freq[5] / CAUSAL_TOTAL, 4)
W[idx[OUTCOME], idx["L6"]] += round(0.2 * layer_total_freq[6] / CAUSAL_TOTAL, 4)

# (g) META LAYER (L7): receives from its own sub-themes only (already set in
#     step (a)) and has NO outgoing edge -- by explicit theoretical
#     requirement, the meta-evidence dimension must not be modeled as a
#     causal contributor to the cascade or the outcome. It is drawn as a
#     dashed, non-quantitative annotation in Figure 1 only (see Section 12).
print(f"\n[NOTE] Layer 7 (meta-evidence) out-degree is fixed at 0 by design: "
      f"it is a non-causal, observational-limitation dimension, per the "
      f"theoretical framework document.")

# (h) special "critical combination" edges documented/derived from the
#     theoretical document's own illustrative examples (kept within the six
#     causal layers only, consistent with the no-meta-causality rule above).
CRITICAL_COMBOS = {
    "مثلث فناورانه-تربیتی-زمانی": [
        "پیچیدگی فنی و بار شناختی–تعاملی فناوری",
        "کاستی دانش و مهارت تخصصی–فناورانه معلم",
        "فشار زمانی، بار اجرایی و تراکم ساختاری",
    ],
    "تلهٔ شناختی-برنامه‌ای": [
        "بار شناختی و دشواری‌های شناختی–فراشناختی",
        "گسست یکپارچگی STEM، برنامه درسی و ارتباط با زمینه واقعی",
    ],
    "سیلوی جنسیتی-فرهنگی": [
        "نابرابری جنسیتی و مشارکت نامتوازن",
        "اختلال در پویایی گروهی و توزیع مشارکت",
        "موانع فرهنگی–اجتماعی و نابرابری دسترسی",
    ],
    "حلقهٔ فرسایش ظرفیت-حمایت": [
        "کسری منابع، زیرساخت و پایداری مالی",
        "ضعف سیاست‌گذاری، راهبری، شبکه پشتیبان و انطباق بافتی",
    ],
}
COMBO_EDGE_WEIGHT = 0.65  # documented synergy weight (higher than ordinary edges)
for combo_name, members in CRITICAL_COMBOS.items():
    for a in members:
        for b in members:
            if a == b:
                continue
            i, j = idx[label_map_sub[a]], idx[label_map_sub[b]]
            if W[i, j] == 0:
                W[i, j] = COMBO_EDGE_WEIGHT

# (i) cross-theme side edges: TF-IDF semantic similarity (the ML/NLP
#     component), only between sub-themes from different CAUSAL layers
#     (the meta-evidence layer, 7, is explicitly excluded from this step
#     too, so it never gains an implicit causal link).
def tokenize_fa(text):
    text = re.sub(r"[^\u0600-\u06FFA-Za-z\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

sub_docs = [" ".join(c["name"] + " " + c["desc"] for c in concepts if c["sub_theme"] == st)
            for st in sub_themes]
vec = TfidfVectorizer(tokenizer=tokenize_fa, lowercase=False, token_pattern=None)
X = vec.fit_transform(sub_docs)
sim = cosine_similarity(X)
sims_flat = sim[np.triu_indices(len(sub_themes), k=1)]
thr = sims_flat.mean() + 1.0 * sims_flat.std()
print(f"\nTF-IDF semantic-similarity threshold for accepting a side edge: {thr:.4f}")

cross_edges = []
for a in range(len(sub_themes)):
    for b in range(len(sub_themes)):
        if a == b:
            continue
        st_a, st_b = sub_themes[a], sub_themes[b]
        lyr_a, lyr_b = sub_row[st_a].layer, sub_row[st_b].layer
        if lyr_a == lyr_b:
            continue
        if lyr_a == META_LAYER or lyr_b == META_LAYER:
            continue  # never create an implicit causal link for the meta layer
        s = sim[a, b]
        if s >= thr:
            i, j = idx[label_map_sub[st_a]], idx[label_map_sub[st_b]]
            if W[i, j] == 0:
                W[i, j] = round(s, 4)  # always positive: both sub-themes are vulnerabilities
                cross_edges.append((st_a, st_b, s, W[i, j]))
print(f"Number of side edges discovered by TF-IDF: {len(cross_edges)}")

pd.DataFrame(W, index=all_nodes, columns=all_nodes).to_csv("table2_weight_matrix.csv", encoding="utf-8-sig")
n_edges = int((W != 0).sum())
density = n_edges / (n * (n - 1))
print(f"Nodes={n} | Edges={n_edges} | Graph density={density:.4f}")



In [ ]:
# %% [8] FCM inference engine (damped Kosko rule) and scenario analysis --------------
LAM, GAMMA = 0.7, 0.1  # selected from the sensitivity analysis (Section 9): stable, non-saturated point

def sigmoid(x, lam=LAM):
    return 1.0 / (1.0 + np.exp(-lam * x))

def run_fcm(A0, clamp_idx=(), clamp_val=(), lam=LAM, gamma=GAMMA, max_iter=300, tol=1e-7):
    A = A0.copy()
    for ci, cv in zip(clamp_idx, clamp_val):
        A[ci] = cv
    history = [A.copy()]
    converged_at = max_iter
    for t in range(max_iter):
        raw = gamma * A + A @ W
        A_new = sigmoid(raw, lam)
        for ci, cv in zip(clamp_idx, clamp_val):
            A_new[ci] = cv
        history.append(A_new.copy())
        if np.linalg.norm(A_new - A) < tol:
            A = A_new; converged_at = t + 1; break
        A = A_new
    return A, np.array(history), converged_at

def clampnodes(labels, val):
    ids = [idx[label_map_sub[l]] for l in labels]
    return ids, [val] * len(ids)

A0 = np.full(n, 0.5)

# Baseline scenario: no intervention, all nodes neutral
A_base, hist_base, it_base = run_fcm(A0.copy())

# Best-case scenario: a well-designed, well-resourced intervention ->
# all sub-theme vulnerabilities are actively suppressed
ci_low, cv_low = clampnodes(sub_themes, 0.15)
A_best, hist_best, it_best = run_fcm(A0.copy(), ci_low, cv_low)

# Worst-case scenario: a completely under-resourced/unplanned intervention
# -> all vulnerabilities are maximally activated
ci_high, cv_high = clampnodes(sub_themes, 0.9)
A_worst, hist_worst, it_worst = run_fcm(A0.copy(), ci_high, cv_high)

# Four critical-combo scenarios (each combo is activated in an otherwise
# neutral context)
combo_scenarios = {}
for combo_name, members in CRITICAL_COMBOS.items():
    ci, cv = clampnodes(members, 0.9)
    A_c, hist_c, it_c = run_fcm(A0.copy(), ci, cv)
    combo_scenarios[combo_name] = (A_c, hist_c, it_c)

scenarios = {
    "Baseline (no intervention)": (A_base, it_base),
    "Best-case (well-resourced intervention)": (A_best, it_best),
    "Worst-case (under-resourced/unplanned)": (A_worst, it_worst),
}
for combo_name, (A_c, _, it_c) in combo_scenarios.items():
    scenarios[combo_name] = (A_c, it_c)

rows = [{"Scenario": name, "Convergence iterations": it,
         "Collapse probability (breakdown point)": round(A[idx[OUTCOME]], 4),
         "Difference from baseline": round(A[idx[OUTCOME]] - A_base[idx[OUTCOME]], 4)}
        for name, (A, it) in scenarios.items()]
df_scn = pd.DataFrame(rows)
print(df_scn.to_string(index=False))
df_scn.to_csv("table3_scenarios.csv", index=False, encoding="utf-8-sig")



In [ ]:
# %% [9] Sensitivity analysis of the inference parameters (λ, γ) ---------------
sens_rows = []
for lam in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    for gamma in [0.05, 0.1, 0.2, 0.3]:
        A_b, _, it_b = run_fcm(A0.copy(), lam=lam, gamma=gamma)
        sens_rows.append({"lambda": lam, "gamma": gamma, "iters": it_b,
                           "collapse_risk_base": round(A_b[idx[OUTCOME]], 4)})
df_sens = pd.DataFrame(sens_rows)
df_sens.to_csv("table4_sensitivity.csv", index=False, encoding="utf-8-sig")
print(df_sens.to_string(index=False))



In [ ]:
# %% [10] Network centrality metrics (transmitter/receiver/ordinary per Kosko's theory) ---------
out_degree = np.sum(np.abs(W), axis=1)
in_degree = np.sum(np.abs(W), axis=0)
rows = []
for k in all_nodes:
    i = idx[k]; od, idg = out_degree[i], in_degree[i]
    role = ("Transmitter" if od > 0 and idg == 0 else
             "Receiver" if od == 0 and idg > 0 else "Ordinary")
    rows.append({"node": k, "label": node_labels[k], "out_degree": round(od, 4),
                 "in_degree": round(idg, 4), "centrality": round(od + idg, 4), "role": role})
df_cent = pd.DataFrame(rows).sort_values("centrality", ascending=False)
df_cent.to_csv("table5_centrality.csv", index=False, encoding="utf-8-sig")
print(df_cent.to_string(index=False))



In [ ]:
# %% [11] Statistical validation: correlation between FCM centrality and raw meta-synthesis frequency -------
sub_cent = df_cent[df_cent["node"].str.startswith("S")]
merged = sub_cent.merge(df_sub, left_on="label", right_on="sub_theme")
r_p, p_p = stats.pearsonr(merged["centrality"], merged["total_freq"])
r_s, p_s = stats.spearmanr(merged["centrality"], merged["total_freq"])
print(f"\nPearson correlation (centrality ~ raw frequency) = {r_p:.4f}  (p={p_p:.4f})")
print(f"Spearman correlation (centrality ~ raw frequency) = {r_s:.4f}  (p={p_s:.4f})")
with open("table6_validation.txt", "w", encoding="utf-8") as f:
    f.write(f"Pearson r={r_p:.4f}, p={p_p:.4f}\nSpearman rho={r_s:.4f}, p={p_s:.4f}\nn={len(merged)}\n")



In [ ]:
# %% [12] Plot figures (Q1 print version) -------------------------------------

# --- 12.1 Short, academic English labels ---
LAYER_LABELS_EN = {
    "L1": "L1: Implementer\n(Teacher) Capacity",
    "L2": "L2: Learner\nVulnerabilities",
    "L3": "L3: Instructional\nDesign & Curriculum",
    "L4": "L4: Technology\nComplexity & Fragility",
    "L5": "L5: Organizational\nResources & Time",
    "L6": "L6: Macro/Contextual\nPolicy & Culture",
    "L7": "L7: Meta-Evidence\n(non-causal)",
    "OUT": "TIPPING POINT\n(Failure / Instability Risk)",
}

# English translations of the 21 new sub-themes. Keys must exactly match the
# raw Persian sub_theme text in the input file. Any sub-theme not covered
# here automatically gets a safe, generic fallback label and a warning.
SUB_LABELS_EN = {
    "کاستی دانش و مهارت تخصصی–فناورانه معلم": "Teacher Tech-Content\nKnowledge Gap",
    "ضعف صلاحیت پداگوژیک و تلفیق برنامه درسی": "Weak Pedagogical\n& Curricular Integration",
    "خودکارآمدی پایین، آمادگی ناکافی و وابستگی به پشتیبانی حرفه‌ای": "Low Self-Efficacy &\nDependence on Support",
    "بار شناختی و دشواری‌های شناختی–فراشناختی": "Cognitive Load &\nMetacognitive Difficulty",
    "ناهمگونی دانش، توانایی و تجربه پیشین": "Prior Knowledge &\nAbility Heterogeneity",
    "فرسایش انگیزشی، عاطفی و نگرشی": "Motivational, Affective\n& Attitudinal Erosion",
    "اختلال در پویایی گروهی و توزیع مشارکت": "Group Dynamics &\nParticipation Disruption",
    "نابرابری جنسیتی و مشارکت نامتوازن": "Gender Inequality &\nUnbalanced Participation",
    "گسست یکپارچگی STEM، برنامه درسی و ارتباط با زمینه واقعی": "STEM Integration &\nReal-World Relevance Gap",
    "عدم تناسب طراحی فعالیت با سطح رشد، توانایی و نیاز یادگیرنده": "Activity-Design/Developmental\nLevel Mismatch",
    "ضعف راهبرد تدریس، داربست‌بندی و سازمان‌دهی اجرا": "Weak Teaching Strategy\n& Scaffolding",
    "پیچیدگی فنی و بار شناختی–تعاملی فناوری": "Technical Complexity &\nInteractive Cognitive Load",
    "نقص، شکنندگی، دشواری عیب‌یابی و قابلیت اطمینان پایین": "Fragility, Faults &\nLow Reliability",
    "ناسازگاری، محدودیت پلتفرم و اصطکاک کاربردپذیری": "Platform Incompatibility\n& Usability Friction",
    "کسری منابع، زیرساخت و پایداری مالی": "Resource, Infrastructure\n& Financial Deficit",
    "فشار زمانی، بار اجرایی و تراکم ساختاری": "Time Pressure &\nStructural Overload",
    "ضعف سیاست‌گذاری، راهبری، شبکه پشتیبان و انطباق بافتی": "Weak Policy, Leadership\n& Support Network",
    "موانع فرهنگی–اجتماعی و نابرابری دسترسی": "Cultural Barriers &\nAccess Inequality",
    "تهدیدهای اعتبار درونی، سوگیری و ضعف طراحی مطالعه": "Internal-Validity Threats\n& Study-Design Bias",
    "کاستی روایی، حساسیت و پوشش ابزارهای سنجش": "Measurement Validity\n& Sensitivity Gaps",
    "ضعف پایش طولی، تبیین سازوکار و قابلیت انتقال شواهد": "Weak Longitudinal Monitoring\n& Transferability",
}

def get_short_label(st):
    if st in SUB_LABELS_EN:
        return SUB_LABELS_EN[st]
    lyr = sub_row[st].layer
    ordinal = [s for s in sub_themes if sub_row[s].layer == lyr].index(st) + 1
    print(f"[WARNING] No English translation found for '{st}'; "
          f"using placeholder label 'Sub-theme {lyr}.{ordinal}'. "
          f"Please add it to the SUB_LABELS_EN dictionary.")
    return f"Sub-theme {lyr}.{ordinal}"

short_labels = dict(LAYER_LABELS_EN)
for st in sub_themes:
    short_labels[label_map_sub[st]] = get_short_label(st)

# --- 12.2 Build the graph ---
G = nx.DiGraph()
for k in all_nodes:
    G.add_node(k)
for i in range(n):
    for j in range(n):
        if W[i, j] != 0:
            G.add_edge(all_nodes[i], all_nodes[j], weight=W[i, j])

# --- 12.3 Figure 1: multilevel layout (L1 core -> L6 outermost causal ring;
#     L7 meta-evidence drawn as a separate satellite cluster, connected to
#     OUT only by a dashed, non-quantitative annotation) ---
pos = {}
layer_radius = {1: 1.6, 2: 3.4, 3: 5.2, 4: 7.0, 5: 8.8, 6: 10.6}
sub_ring_offset = 1.1

pos["L1"] = (0.0, 0.0)  # central core
for lyr in range(2, 7):
    pos[f"L{lyr}"] = (0.0, layer_radius[lyr])

sub_by_layer = {lyr: [st for st in sub_themes if sub_row[st].layer == lyr] for lyr in range(1, 8)}
for lyr in range(1, 7):
    members = sub_by_layer[lyr]
    r = layer_radius[lyr] + sub_ring_offset if lyr > 1 else layer_radius[1] + sub_ring_offset
    k_n = len(members)
    for m_i, st in enumerate(members):
        ang = np.pi / 2 + 2 * np.pi * (m_i / max(k_n, 1))
        pos[label_map_sub[st]] = (r * np.cos(ang), r * np.sin(ang))

pos[OUTCOME] = (0.0, -(layer_radius[6] + sub_ring_offset + 2.5))  # bottom of the figure

# L7 (meta) satellite cluster: placed off to the side, clearly separated
meta_x = layer_radius[6] + 6.0
pos["L7"] = (meta_x, 2.0)
meta_members = sub_by_layer[7]
for m_i, st in enumerate(meta_members):
    pos[label_map_sub[st]] = (meta_x + 2.4, 2.0 + (m_i - (len(meta_members) - 1) / 2) * 2.2)

node_colors, node_sizes = [], []
meta_sub_ids = {label_map_sub[st] for st in meta_members}
for k in all_nodes:
    if k == OUTCOME:
        node_colors.append("#C44E52"); node_sizes.append(4200)
    elif k == "L7":
        node_colors.append("#8C8C8C"); node_sizes.append(3200)
    elif k.startswith("L"):
        node_colors.append("#4C72B0"); node_sizes.append(3200)
    elif k in meta_sub_ids:
        node_colors.append("#B0A8B9"); node_sizes.append(2000)
    else:
        node_colors.append("#DD8452"); node_sizes.append(2000)

fig, ax = plt.subplots(figsize=(24, 18))
edge_widths = [1.5 + 6 * abs(G[u][v]["weight"]) for u, v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax,
                        edgecolors="black", linewidths=2)
nx.draw_networkx_edges(G, pos, edge_color="#7A1F1F", width=edge_widths, arrows=True,
                        arrowsize=20, connectionstyle="arc3,rad=0.08", ax=ax, alpha=0.55)
nx.draw_networkx_labels(G, pos, labels={k: short_labels[k] for k in all_nodes},
                         font_size=11, ax=ax)

# Dashed, non-quantitative annotation for the meta-evidence dimension's
# epistemic (not causal) relationship to the outcome node.
ax.annotate(
    "", xy=pos[OUTCOME], xytext=pos["L7"],
    arrowprops=dict(arrowstyle="->", linestyle="dashed", color="#555555", lw=1.5,
                     connectionstyle="arc3,rad=-0.3"),
)
mid_x = (pos[OUTCOME][0] + pos["L7"][0]) / 2
mid_y = (pos[OUTCOME][1] + pos["L7"][1]) / 2
ax.text(mid_x, mid_y, "limits observability,\nnot a causal path",
        fontsize=10, color="#555555", ha="center", style="italic")

ax.set_title("Fuzzy Cognitive Map — Multilevel Failure & Instability Model (v2)",
             fontsize=16, pad=20)
ax.axis("off")
plt.tight_layout(pad=4)
save_fig(fig, "fig1_fcm_network")

# --- 12.4 Figure 2: weight heatmap ---
fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(W, cmap="Reds", vmin=0, vmax=1, interpolation="nearest")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([short_labels[k] for k in all_nodes], rotation=90, fontsize=10)
ax.set_yticklabels([short_labels[k] for k in all_nodes], fontsize=10)
cbar = plt.colorbar(im, ax=ax, label="Edge weight (vulnerability propagation strength)")
cbar.ax.tick_params(labelsize=12)
cbar.set_label("Edge weight (vulnerability propagation strength)", fontsize=13)
ax.set_title("FCM Adjacency (Weight) Matrix — Multilevel Model (v2)", fontsize=16, pad=20)
plt.tight_layout(pad=4)
save_fig(fig, "fig2_weight_heatmap")

# --- 12.5 Figure 3: scenario comparison ---
COMBO_LABELS_EN = {
    "مثلث فناورانه-تربیتی-زمانی": "Techno-Pedagogical-\nTemporal Triangle",
    "تلهٔ شناختی-برنامه‌ای": "Cognitive-Curricular Trap",
    "سیلوی جنسیتی-فرهنگی": "Gender-Cultural Silo",
    "حلقهٔ فرسایش ظرفیت-حمایت": "Capacity-Support\nErosion Loop",
}
scn_en = ["Baseline", "Best-case\n(well-resourced)", "Worst-case\n(under-resourced)"] + \
         [f"Combo:\n{COMBO_LABELS_EN.get(name, name)}" for name in CRITICAL_COMBOS.keys()]
vals = df_scn["Collapse probability (breakdown point)"].tolist()
colors_scn = ["#4C72B0", "#2E8B57", "#B22222", "#DD8452", "#8172B2", "#937860", "#CCB974"]
fig, ax = plt.subplots(figsize=(15, 8.5))
bars = ax.bar(scn_en, vals, color=colors_scn[:len(vals)], edgecolor="black", linewidth=1.5)
ax.axhline(vals[0], color="gray", linestyle="--", linewidth=1.5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.4f}", ha="center", fontsize=12)
ax.set_ylabel("Steady-state failure/instability risk\n(Tipping Point activation)", fontsize=13)
ax.set_ylim(0, 1.0)
ax.set_title("FCM Scenario Analysis — Multilevel Model (v2)", fontsize=16, pad=20)
plt.xticks(rotation=15, ha="right", fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout(pad=4)
save_fig(fig, "fig3_scenario_comparison")

# --- 12.6 Figure 4: convergence curves ---
fig, ax = plt.subplots(figsize=(14, 8.5))
hist_list = [hist_base, hist_best, hist_worst] + [combo_scenarios[c][1] for c in CRITICAL_COMBOS]
for h, lab, col in zip(hist_list, scn_en, colors_scn[:len(hist_list)]):
    ax.plot(range(len(h)), h[:, idx[OUTCOME]], marker="o", markersize=5,
            linewidth=2, label=lab.replace("\n", " "), color=col)
ax.set_xlabel("Iteration", fontsize=13)
ax.set_ylabel("Activation of TIPPING POINT node", fontsize=13)
ax.set_title("Convergence Trajectories across Scenarios (v2)", fontsize=16, pad=20)
ax.legend(fontsize=11)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout(pad=4)
save_fig(fig, "fig4_convergence")

# --- 12.7 Figure 5: centrality ---
df_cent_sorted = df_cent.sort_values("centrality", ascending=True)
fig, ax = plt.subplots(figsize=(13, 14))
y = range(len(df_cent_sorted))
ax.barh(y, df_cent_sorted["out_degree"], color="#2E8B57", label="Out-degree")
ax.barh(y, df_cent_sorted["in_degree"], left=df_cent_sorted["out_degree"],
        color="#4C72B0", label="In-degree")
ax.set_yticks(y)
ax.set_yticklabels([short_labels[k] for k in df_cent_sorted["node"]], fontsize=10)
ax.set_xlabel("Centrality", fontsize=13)
ax.set_title("Node Centrality Decomposition — Multilevel Model (v2)", fontsize=16, pad=20)
ax.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.tight_layout(pad=4)
save_fig(fig, "fig5_centrality")

# --- 12.8 Figure 6: sensitivity ---
fig, ax = plt.subplots(figsize=(11.5, 8))
for gamma in sorted(df_sens["gamma"].unique()):
    sub = df_sens[df_sens["gamma"] == gamma]
    ax.plot(sub["lambda"], sub["collapse_risk_base"], marker="o", markersize=6,
            linewidth=2, label=f"γ={gamma}")
ax.set_xlabel("Sigmoid steepness (λ)", fontsize=13)
ax.set_ylabel("Baseline steady-state failure/instability risk", fontsize=13)
ax.set_title("Sensitivity Analysis of FCM Parameters — Multilevel Model (v2)", fontsize=16, pad=20)
ax.legend(title="Memory coeff.", fontsize=11, title_fontsize=12)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout(pad=4)
save_fig(fig, "fig6_sensitivity")

print(f"\nAll six figures (each as a {DPI}-dpi PNG plus a vector PDF) "
      "and six tables were generated and saved successfully.")



In [ ]:
# %% [13] (Optional) LLM-assisted semantic edge-weight refinement module (Claude API) ---
# This section only activates if an API key is present, and acts as a
# secondary semantic-validation layer (not a replacement for the
# statistical weights). Running it is optional.
"""
import os, requests
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if API_KEY:
    def llm_edge_check(sub_a, sub_b):
        prompt = (f"In a model of cascading failure for educational robotics interventions, "
                  f"rate how strongly vulnerability A intensifies vulnerability B on a scale "
                  f"from 0 (unrelated) to 1 (very strong intensification). "
                  f"Respond with ONLY a number.\nA: {sub_a}\nB: {sub_b}")
        r = requests.post("https://api.anthropic.com/v1/messages",
                           headers={"x-api-key": API_KEY, "anthropic-version": "2023-06-01",
                                    "content-type": "application/json"},
                           json={"model": "claude-sonnet-4-6", "max_tokens": 10,
                                 "messages": [{"role": "user", "content": prompt}]})
        return r.json()
    # Example usage (disabled by default):
    # print(llm_edge_check(sub_themes[0], sub_themes[5]))
else:
    print("Note: ANTHROPIC_API_KEY is not set; the LLM semantic-refinement "
          "module was skipped (the paper's main results are based purely "
          "on TF-IDF and the actual empirical statistics).")
"""
